# 第 5 章 · 上下文工程与 Prompt Cache

> **学习材料**：《Harness Engineering》第 5 章（书页 103–129，共 27 页）
> **配套代码**：[harness-py-book](https://github.com/Luxuzhou/harness-py-book)（`examples/ch05/`）

## 📌 本章开篇要回答的 3 个问题

书中在章首明确提出，读完本章应能回答：

| # | 问题 | 对应小节 |
|---|------|----------|
| ① | Agent 在执行操作前，**应该先看哪些信息？哪些信息不应该进入上下文？** | 5.1 |
| ② | 如何组织仓库文档、系统提示词和工具描述，才能让**模型更快收敛到正确决策**？ | 5.2 / 5.3 |
| ③ | Prompt Cache 为什么**不是成本优化的"小技巧"，而是上下文工程的基础设施**？ | 5.4 |

> 💡 记忆锚点：**选信息（5.1）→ 摆位置（5.2/5.3）→ 省算力（5.4）**。

## 0. 先导概念：为什么"上下文"值得单独一章

Agent 每一轮都在消费上下文窗口（200K Token 级别），而窗口是**稀缺资源**，三个问题随之而来：

- **放什么**：放错了，模型会学会坏习惯、产生错误行为（5.1 的实验数据）；
- **怎么放**：同样的内容，层级和拼接位置不同，模型吸收效率完全不同（5.2 / 5.3）；
- **怎么省**：重复前缀每轮都重新计费，不缓存则成本随轮数线性放大（5.4）。

一句话：**上下文工程 = 为模型策展信息；Prompt Cache = 让策展成果可以跨轮复用**。

## 5.1 什么信息应该进入上下文

**核心命题（5.1.1）**：Agent 的行为质量由它"看到"的环境线索决定。上下文窗口就是模型的工作环境——环境里有什么样的代码与指令，模型就长成什么样。

### 5.1.1 环境线索决定行为质量

- 模型不是"规则执行器"，而是 **"模仿者"** ：上下文中的代码风格、错误处理方式、命令习惯都会被它统计性地继承；
- 由此得出上下文工程第一定律：**往上下文里放的东西，等于在教模型"这个项目里正确做法长这样"**；
- 反面推论：放了坏示例（`os.system`、`print` 调试、裸 `except`……），模型会照单全收——见表 5-1 的实验。

### 5.1.2 精简 AGENTS.md：该写进来的关键信息

书中将 AGENTS.md / CLAUDE.md 的关键信息归纳为 4 类，正文重点展开以下 3 类：

| 类别 | 典型内容 | 为什么必须写 |
|------|---------|-------------|
| ① 构建与测试命令 | lint / unit test / build 的**确切命令** | Agent 依赖这些命令完成"修改 → 验证"闭环；不写就会自行猜测，浪费轮次甚至编造命令 |
| ② 代码规范与架构约束 | 分层规则（如 Controller 禁止直接操作数据库） | 模型默认按"开源社区常见写法"生成，与项目架构冲突时必须显式禁止 |
| ③ 明确命令与替代路径 | 指定唯一命令，并明确禁用替代方案 | 替代方案在开源代码中更常见，模型会**概率性"改用"** 它们 |

> 页面上还引用了 Anthropic 内部使用数据作佐证：把验证命令写进 CLAUDE.md，能显著减少无效轮次。

### 表 5-1：环境线索的"传染性"实验

向上下文注入不同倾向的示例代码，统计模型产出对应反模式的比率（数值照录原书，两列对应两种上下文条件，具体实验设置以原书为准）：

| 行为类别 | 条件一 | 条件二 |
|----------|--------|--------|
| `os.system` 类调用 | 100% | 0% |
| `print` 调试 | 100% | 0% |
| `eval()` 类调用 | 70% | 100% |
| 裸 `except` | 0% | 0% |
| SQL 注入 | 0% | 0% |

**解读**：`os.system`、`print` 调试这类"坏习惯"对上下文高度敏感——环境里出现过就大量复现；而 SQL 注入这类在训练数据中反复见过反例的行为，几乎不受上下文影响。

> **上下文是放大器，不是开关**：它强化既有倾向，而不是凭空创造行为。

### 表 5-2：文档越多越好吗？

实验设置：DeepSeek 模型，3 个独立 Bug 修复任务，每组跑 5 次取均值：

| 级别 | 文档规模 | 文档 Token 数 | 完成率 | 平均轮数 | 平均消耗 Token |
|------|---------|--------------|--------|---------|---------------|
| L0 | 无文档 | 0 | 100% | 10.9 | 38,324 |
| L1 | 精简 | 109 | 100% | 10.5 | 38,409 |
| L2 | 标准 | 525 | 100% | 11.0 | 39,703 |
| L3 | 扩展 | 2,970 | 100% | 11.5 | 67,711 |

**结论**：
- 完成率全部 100%——**文档多少不决定"能不能做"**；
- L3 扩展文档让 Token 消耗暴涨约 76%（38K → 68K），平均轮数反而最多；
- L1 精简文档只花 109 个 Token，轮数最少、总消耗几乎不变。

> **CLAUDE.md 应该是"目录"，不是"百科全书"**——这句话在 5.3 节的 Token 预算表（表 5-4）里会再次得到定量印证。

### 5.1.3 学术佐证：AGENTS.md 的量化收益

arXiv 论文（AGENTS.md 影响研究）实测，为仓库接入 AGENTS.md 后：

- **Token 消耗下降 28.6%**
- **Counterpoint（模型输出中的对立观点 / 风险陈述）增加 16.6%**
- **总体反例（negative cases）下降约 20%**

> 解读：文档带来的不是"更啰嗦"，而是**更少的试错轮次**（省 Token）与**更审慎的输出**（更多风险提示、更少错误案例）。

### 表 5-3：不该放进 CLAUDE.md 的内容——迁移清单

| CLAUDE.md 中的内容 | 应迁移到 | 迁移理由 |
|--------------------|---------|---------|
| 完整 API 清单 | `docs/<resource>-api.md` | 接口数量多、变化快，常驻上下文性价比极低 |
| 数据库表结构 | `schema.md` | 属于"参考数据"而非"操作指令" |
| 详细业务逻辑 | `docs/design/` | 变化频繁，写死很快过时 |
| 超过 10 行的代码示例 | `examples/` | 内联编辑难度大，且随版本迭代失真 |
| 历史决策记录（ADR） | `docs/decisions/` | 按需获取即可，不必每次装载 |
| 个人偏好（本地端口、API Token 等） | `CLAUDE.local.md` | 敏感信息，不应进入版本控制 |

**判断标准一句话**：CLAUDE.md 只放**每次会话都要用的高频指令**；其余内容给"指针"，让 Agent 按需读取。

### 📎 5.1 小结

1. 上下文是"环境"，模型是"模仿者"——放什么进去，它就长成什么样（表 5-1）；
2. 文档贵精不贵多：L1 精简文档以最小成本达到同等完成率（表 5-2）；
3. 学术侧验证：接入 AGENTS.md 后 Token −28.6%、反例 −20%（5.1.3）；
4. 用表 5-3 定期"瘦身"：CLAUDE.md 只留高频指令，其余迁移或给指针。

## 5.2 配置文件的分层架构

### 图 5-1：三层配置金字塔（书中插图复现）

![图 5-1 三层配置金字塔](../src/context-three-layer-docs.svg)

| 层级 | 位置 | 管辖范围 | 特点 |
|------|------|---------|------|
| 全局层 | `~/.claude/` | 本机全部项目 | 最抽象、最通用 |
| 项目层 | 项目根 `CLAUDE.md` / `AGENTS.md` | 当前项目 | 团队共享、进版本控制 |
| 任务层 | 子目录规则、模块约束 | 特定模块 / 任务 | 最具体、约束最强 |

- **越往上越抽象通用，越往下越具体、约束越强**；
- Context Builder（5.3 详解）按这套层级发现文件、合并内容，拼出最终系统提示词；
- 与 5.1 呼应：三层设计正是"精简"原则的空间化——**每层只放该层才能回答的问题**。

### 5.2.2 辅助配置：CLAUDE.local.md 与 .claude/rules/

**CLAUDE.local.md（个人本地覆盖）**
- 不提交版本控制（写入 `.gitignore`）；
- 适合：临时实验、个人端口 / 路径偏好、与团队规范暂不一致的本地调整。

**`.claude/rules/` 目录（条件化规则）**
- 把低频规则拆成独立文件（如 `security.md` 只在涉及认证 / 安全代码时生效）；
- Agent 启动时扫描目录、按需注入，避免所有规则每轮全量装载。

**层级冲突实测**：即使全局层与项目层都使用 `MUST` 强制描述，模型仍有 **5%~10% 概率遵守全局层而非项目层**。可靠写法是在项目层显式声明覆盖，例如：*"本项目使用 unittest 而非 pytest（覆盖全局设定）"*。

### 核心代码：discover_claude_md（依原书整理）

函数从当前目录开始**向上遍历**，逐层查找 `CLAUDE.md`，并沿同一路径收集每层的 `.claude/rules/`。这样在 monorepo（单体仓库）里，子项目可以有自己的规则，同时继承根目录通用规则，不必重复存放在每个子目录。找到文件后读取内容，通过 `seen` 集合将重复路径去重，最终按发现顺序返回列表，组装系统提示词时按此顺序注入。

In [1]:
from pathlib import Path


def discover_claude_md(cwd: Path) -> list[Path]:
    '''发现项目中所有 CLAUDE.md / AGENTS.md 文件（依原书页 111 整理）'''
    candidates: list[Path] = []
    seen: set[Path] = set()
    cwd = Path(cwd)

    # 1) 向上遍历父目录：逐层查找 CLAUDE.md / AGENTS.md
    for parent in [cwd, *cwd.resolve().parents]:
        for name in ("CLAUDE.md", "AGENTS.md"):
            candidate = parent / name
            if candidate.exists() and candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    # 2) 全局层：~/.claude/CLAUDE.md
    home_claude = Path.home() / ".claude" / "CLAUDE.md"
    if home_claude.exists() and home_claude not in seen:
        seen.add(home_claude)
        candidates.append(home_claude)

    # 3) Rules 目录：当前目录 + 所有父级的 .claude/rules/
    rules_dirs = [cwd / ".claude" / "rules"]
    for parent in cwd.resolve().parents:
        rules_dirs.append(parent / ".claude" / "rules")
    for rd in rules_dirs:
        if rd.is_dir():
            candidates.extend(sorted(rd.glob("*.md")))

    return candidates


print(f"当前环境发现 {len(discover_claude_md(Path.cwd()))} 个配置文件")

当前环境发现 0 个配置文件


> 🔑 **伏笔**：`sorted(rd.glob("*.md"))` 这一行看似多余，实则是 5.4 节缓存命中率的关键——文件系统返回顺序不确定，一旦顺序变化，整个静态前缀的缓存全部失效。
>
> **上下文工程的一切"确定性偏执"，都是在为缓存铺路。**

### 5.2.3 分层实践的目录规划

书中思路（详见原书）：把 5.1 的"精简"原则落到目录结构上——

- 根目录 `CLAUDE.md`：只放全项目高频指令（构建 / 测试 / 架构红线）；
- `docs/`、`examples/`、`schema.md`：参考资料，供 Agent 按需读取（表 5-3 的迁移目标）；
- `.claude/rules/`：条件化规则，按任务注入；
- `CLAUDE.local.md`：个人本地覆盖，不进版本控制。

### 5.2.4 案例：Express.js 项目的 CLAUDE.md 四处关键设计

| # | 设计 | 手法 | 解决的问题 |
|---|------|------|-----------|
| ① | 用命令压缩消除行为偏差 | 把"装依赖 → 迁移 → 启动 → 验证"压成一条命令 | 验证成本从分钟级降到秒级，模型才真的**每次都验证** |
| ② | 用先验指令编排任务步骤 | 显式规定 migrate 必须先于 dev 执行 | 防止模型自由发挥执行顺序 |
| ③ | 主动设置反模式警示 | 明确禁止"在 Express 路由里直接调 `prisma client.findMany()`" | 架构红线写成禁令，而非留给模型"领会" |
| ④ | 关闭取巧逃生通道 | TypeScript 全量禁止 `any` | 不给模型留"类型偷懒"出口 |

配套的 TypeScript 类型规范：严格模式、显式返回类型、禁止 `any`——把"规范"写成**机器可检查的硬约束**，而非口头约定。

### 📎 5.2 小结

1. 三层金字塔给"信息放哪"一个空间答案：全局层通用、项目层共享、任务层精准（图 5-1）；
2. 辅助配置让"精简"可持续：`local` 覆盖个人偏好，`rules` 目录条件化低频规则；层级冲突要靠**项目层显式声明覆盖**解决；
3. `discover_claude_md` 的 `sorted` 不是洁癖，是为缓存命中率买保险（5.4 伏笔）；
4. Express.js 案例四处设计的共同本质：**用确定性的指令消灭模型的不确定性**。

## 5.3 系统提示词组装：把好钢用在刀刃上

5.1 与 5.2 解决了"信息从哪来"，本节解决"信息怎么进上下文"——**系统提示词的组装**。核心矛盾是：上下文窗口虽大（200K），但注意力是稀缺资源，塞得越满，模型对关键规则的遵循率越低。

![图 5-2 系统提示词组装流程](../src/system-prompt-assembly.svg)

*图 5-2：系统提示词组装流程——多个来源按固定区段拼装，右侧标注各部分的 Token 预算（对应表 5-4）*

### 5.3.1 五区段结构：稳定前缀 + 动态后缀

原书把生产级系统提示词划分为五个区段，顺序固定：

| 区段 | 内容 | 来源 | 变化频率 |
|---|---|---|---|
| ① 基础规则 | 安全红线、输出格式、通用行为约束 | 全局层配置 / 平台默认 | 几乎不变 |
| ② 项目上下文 | 项目背景、技术栈、架构约束 | `CLAUDE.md` / `AGENTS.md` | 极少变化 |
| ③ 任务规则 | 领域规则（如安全扫描规范） | 条件注入的规则文件 | 按任务类型变化 |
| ④ 工具定义 | 工具 schema 与描述 | 工具注册表 | 稳定 |
| ⑤ 当前任务 | 用户输入、日期、工作目录 | 运行时注入 | 每次请求都变 |

**关键洞察**：①~④ 是长而稳定的前缀，⑤ 是短而多变的后缀。这个结构在 5.4 节会被证明价值千金——稳定前缀正是 Prompt Cache 的命中区域。

### 5.3.2 Token 预算：给每一块上下文"定量"

原书给出一套 128K 窗口下的预算分配（表 5-4），并在图 5-2 中以右侧标注呈现：

**表 5-4 上下文窗口 Token 预算分配**

| 组成部分 | Token 预算 | 占比 |
|---|---|---|
| 系统提示词 | 12,800 | 10% |
| 历史会话 | 64,000 | 50% |
| 工具返回 | 25,600 | 20% |
| 输出预留 | 19,200 | 15% |
| Memory | 6,400 | 5% |

三个设计意图：

- **历史会话占一半（50%）**：对话连续性是 Agent 任务质量的生命线，宁可牺牲其他部分。
- **工具返回占 20%**：工具输出（如日志、搜索结果）最容易失控膨胀，必须设上限。
- **输出预留 15%**：不留余量，长回答会被窗口硬截断。

**单文件 40_000 字符截断**：读取大文件时硬编码 `content[:40_000]`——约 10,000 Token，只占 200K 窗口的 5%、128K 窗口的 8%。一个失控的 200KB 日志文件就能吃掉整个预算的 1/3，截断是防单点爆炸的保险丝。

### 5.3.3 条件注入：按需加载规则文件

不是所有规则都该常驻上下文。原书用 **YAML front matter** 给规则文件打标签，只在任务命中关键词时才注入：

In [2]:
# 条件注入示例（依原书页 116 整理）：规则文件带 front matter 元数据
SECURITY_RULES = '''---
applies_to: ["auth", "security", "jwt", "password", "token"]
priority: high
---
# 安全开发规范
- 禁止明文存储密码，必须使用 bcrypt/argon2
- JWT 密钥必须从环境变量读取
- 所有输入必须校验后再进 SQL 语句
'''


def should_inject(rules: str, task_keywords: list[str]) -> bool:
    # 解析 front matter 的 applies_to，判断本次任务是否命中
    meta_line = rules.split('---')[1]            # 取 front matter 主体
    applies_line = next(
        l for l in meta_line.splitlines() if l.startswith("applies_to")
    )
    applies = [
        kw.strip().strip('"')                    # ["auth","security",...] -> 列表
        for kw in applies_line.split("[")[1].split("]")[0].split(",")
    ]
    return any(kw in applies for kw in task_keywords)


print(should_inject(SECURITY_RULES, ["jwt", "login"]))   # True  -> 注入
print(should_inject(SECURITY_RULES, ["css", "layout"]))  # False -> 跳过

True
False


### 5.3.4 `build_system_prompt`：生产版组装器

原书的生产版在基础上增加了两个参数：**`role_prompt`**（多 Agent 编排时给不同角色不同的提示词）与 **`scan_threats`**（安全 Agent 才开启威胁扫描前缀）：

In [3]:
from pathlib import Path
import datetime


def build_system_prompt(
    project_root: Path,
    role_prompt: str | None = None,   # 多 Agent 编排：不同角色不同前缀
    scan_threats: bool = False,       # 安全 Agent 才开启
    task_keywords: list[str] | None = None,
) -> str:
    sections: list[str] = []

    # ① 头部：普通任务 vs 安全扫描任务
    sections.append(
        "⚠ 安全警告：本次任务涉及敏感操作，先做威胁建模再动手"
        if scan_threats else "# 项目规则"
    )

    # ② 角色提示词（多 Agent 编排）
    if role_prompt:
        sections.append(role_prompt)

    # ③ 基础规则 + 项目上下文（来自 CLAUDE.md，长而稳定）
    sections.append("技术栈：Python 3.11 / FastAPI / SQLite")
    sections.append("测试框架：unittest（禁止 pytest）")

    # ④ 条件注入的任务规则
    if task_keywords and should_inject(SECURITY_RULES, task_keywords):
        sections.append(SECURITY_RULES.split('---')[2].strip())

    # ⑤ 动态后缀：日期 + 工作目录
    sections.append(
        f"当前日期：{datetime.date.today().isoformat()}\n"
        f"工作目录：{project_root}"
    )
    return "\n\n".join(sections)


print(build_system_prompt(Path.cwd(), scan_threats=True, task_keywords=["jwt"])[:300])

⚠ 安全警告：本次任务涉及敏感操作，先做威胁建模再动手

技术栈：Python 3.11 / FastAPI / SQLite

测试框架：unittest（禁止 pytest）

# 安全开发规范
- 禁止明文存储密码，必须使用 bcrypt/argon2
- JWT 密钥必须从环境变量读取
- 所有输入必须校验后再进 SQL 语句

当前日期：2026-10-05
工作目录：D:\dev\Developing-LLM-Powered-Translation-Applications\harness-engineering


### 5.3.5 动静分离：为缓存而生的提示词结构

再看上面组装器：①~④ 在一次会话里基本不变（**静态**），⑤ 的日期与工作目录每次启动都变（**动态**）。如果把它们揉在一个字符串里，任何字节变化都会污染整个前缀的缓存资格。

原书的做法是显式插入**边界标记**，把静态与动态部分隔开：

In [4]:
# 动静分离（依原书页 118 整理）：边界标记只存在于组装阶段
_SYSTEM_PROMPT_DYNAMIC_BOUNDARY = "__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__"


def build_system_prompt_with_boundary(cwd: Path) -> str:
    static_part = "\n\n".join([
        "# 项目规则",
        "技术栈：Python 3.11 / FastAPI / SQLite",
        "测试框架：unittest（禁止 pytest）",
    ])
    dynamic_part = "\n\n".join([
        f"当前日期：{datetime.date.today().isoformat()}",   # 每天变化
        f"工作目录：{cwd}",                                  # 每个项目变化
    ])
    return f"{static_part}\n\n{_SYSTEM_PROMPT_DYNAMIC_BOUNDARY}\n\n{dynamic_part}"


prompt = build_system_prompt_with_boundary(Path.cwd())
print("边界标记前的静态部分：")
print(prompt.split(_SYSTEM_PROMPT_DYNAMIC_BOUNDARY)[0])

边界标记前的静态部分：
# 项目规则

技术栈：Python 3.11 / FastAPI / SQLite

测试框架：unittest（禁止 pytest）




![图 5-3 缓存边界机制](../src/cache-boundary-mechanism.svg)

*图 5-3：缓存边界机制——边界标记把系统提示词切成静态/动态两段*

**边界标记去哪了？** 它不会出现在最终发给模型的消息里。API 客户端在发送前检测到边界标记，把它替换为 Anthropic 的 `cache_control` 参数——静态段获得可缓存资格，动态段保持普通文本。具体的替换逻辑就是 5.4 节的 `apply_cache_control` 函数。

### 5.3 小结

- **五区段结构**是系统提示词的"装配图纸"：稳定前缀（基础规则/项目上下文/任务规则/工具定义）在前，动态后缀（当前任务）在后。
- **Token 预算**（表 5-4）防止任何单一来源挤爆窗口：历史会话 50%、工具返回 20%、输出预留 15%。
- **40_000 字符截断**是单文件保险丝，约占窗口 5%~8%。
- **条件注入**用 YAML front matter 按需加载规则，避免无关规则常驻上下文。
- **`build_system_prompt`** 用 `role_prompt` 与 `scan_threats` 支持多 Agent 编排。
- **动静分离**把"明天还会用到的部分"与"每次都变的部分"切开，边界标记最终被替换为 `cache_control`——为 5.4 节的 Prompt Cache 铺平道路。

## 5.4 Prompt Cache：上下文工程的基础设施

前三节都在努力把上下文"做对"——本节回答一个成本与速度问题：**同样的前缀为什么要付两遍钱？**

Anthropic 于 2024 年底推出 Prompt Cache 机制。它不是一个小技巧，而是上下文工程的基础设施：Agent 的系统提示词动辄上万 Token，50 轮对话如果每轮都全价重算，既慢又贵。缓存命中后，读取价格远低于常规输入，50 轮会话累计可节省数十秒等待时间。

### 5.4.1 KV 缓存的工作原理

Transformer 推理时会把每个 Token 的 Key/Value 中间态存下来（KV Cache）。如果本次请求的**前缀**与上次完全一致，这些中间态可以直接复用，跳过整段前缀的重算。

四个关键性质（依原书页 122 整理）：

- **"完全相同"是严格逐字节匹配**：哪怕只改一个空格、一个 JSON 键的顺序，前缀从此处起的所有 KV 中间态都无法复用。
- **首轮写入，后续摊薄**：第一次请求没有命中，整段前缀按（略高的）写入价计算；从第二轮起开始摊薄成本。
- **前缀越长越省**：可复用的中间态越多，命中带来的节省越显著。
- **4096 Token 最小缓存块**：缓存按块计量，前缀不足一个块无法命中；这也意味着把稳定内容尽量堆在前面才有意义。

### 5.4.2 `apply_cache_control`：边界标记的消费者

5.3.5 节埋下的边界标记 `__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__` 在这里被消费：静态段打上 `cache_control` 标签获得可缓存资格，动态段保持普通文本。**边界标记本身不会发给模型**（依原书页 123 原文逻辑）：

In [5]:
# apply_cache_control（依原书页 122~123 整理）
_SYSTEM_PROMPT_DYNAMIC_BOUNDARY = "__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__"


def apply_cache_control(system_prompt: str) -> list[dict]:
    # 无边界标记：整段按普通文本处理（无法精细缓存）
    if _SYSTEM_PROMPT_DYNAMIC_BOUNDARY not in system_prompt:
        return [{"type": "text", "text": system_prompt}]

    # 有边界标记：切成静态/动态两段
    static_part, dynamic_part = system_prompt.split(_SYSTEM_PROMPT_DYNAMIC_BOUNDARY)

    return [
        {
            "type": "text",
            "text": static_part,
            "cache_control": {"type": "ephemeral"},   # 静态段 -> 可缓存
        },
        {
            "type": "text",
            "text": dynamic_part,                      # 动态段 -> 每次重算
        },
    ]


demo = "# 规则\n稳定前缀" + _SYSTEM_PROMPT_DYNAMIC_BOUNDARY + "\n今天：2026-10-03"
for block in apply_cache_control(demo):
    print({k: v for k, v in block.items() if k != "text"}, "<-文本长度", len(block["text"]))

{'type': 'text', 'cache_control': {'type': 'ephemeral'}} <-文本长度 9
{'type': 'text'} <-文本长度 14


### 5.4.3 静态前缀最大化：三条工程纪律

命中率的敌人是"任何位置的字节漂移"。原书给出三条建议（页 123）：

1. **消息顺序确定性**：`discover_claude_md`（5.2.2 的代码）返回的文件列表顺序必须稳定——任何一次文件发现顺序的变化都会让前缀错位，缓存全灭。这正是该函数对 `rules/*.md` 使用 `sorted()` 的原因（当时埋下的伏笔在此兑现）。
2. **JSON 键排序**：`json.dumps` 默认 `sort_keys=False`，序列化顺序跟随 dict 插入序——不同进程、不同版本可能产生不同字节流。**始终 `sort_keys=True`**。
3. **`ensure_ascii=False` + 增量替换**：默认转义会把中文变成 `\uXXXX` 序列，既费 Token 又制造字节差异；对话历史用增量替换（只追加新消息）而非全量替换。

In [6]:
import json

tool_args = {"token": "abc", "password": "***", "auth": True}

print("默认（键序跟随插入序）:", json.dumps(tool_args))
print("排序后（字节流稳定） :", json.dumps(tool_args, sort_keys=True))
print("中文不转义           :", json.dumps({"说明": "校验密码"}, ensure_ascii=False))
print("中文被转义           :", json.dumps({"说明": "校验密码"}))

默认（键序跟随插入序）: {"token": "abc", "password": "***", "auth": true}
排序后（字节流稳定） : {"auth": true, "password": "***", "token": "abc"}
中文不转义           : {"说明": "校验密码"}
中文被转义           : {"\u8bf4\u660e": "\u6821\u9a8c\u5bc6\u7801"}


### 5.4.4 命中率实测：动态内容放哪里？

原书用五组对照实验量化"动态内容位置"的影响（表 5-5，页 124）：

**表 5-5 缓存命中率实测**

| 方案 | 动态内容位置 | 命中率 |
|---|---|---|
| A 基线 | 无动态内容 | 88.6% |
| B 秒级时间戳 | 放在前缀头部 | 0% |
| C Request-ID | 放在前缀头部 | 0% |
| D 时间戳（只保留 MM-DD） | 放在前缀尾部 | 89.0% |
| E 工具随机排序 | 工具定义区 | 82.9% |

三个结论：

- **动态内容放前缀头部 = 缓存死刑**（B/C 直接归零）：从第一个字节不同处起，后面全部作废。
- **时间戳精度降到"天"并挪到尾部**（D），命中率反而略高于基线——因为静态前缀足够长且稳定。
- **工具随机排序**（E）同样致命：工具定义属于静态前缀的一部分，顺序抖动损失 6 个百分点。
- 另一个实验细节：**会话前 5 轮可共享同一份缓存**——这意味着保活与复用策略在第 5 轮前收益最大。

### 5.4.5 命中率调优：五个工程方法（页 124~125）

1. **工具 Schema 排序**：注册工具后按名称排序，消除注册顺序的字节抖动：

```python
sorted(schemas, key=lambda s: s["name"])
```

2. **时间戳位置**：动态字段一律放前缀尾部；需要日期时精度降到 `MM-DD`（表 5-5 的 0% vs 89.0%）。
3. **MCP Server 按名称排序**：多 MCP Server 的工具聚合顺序同样要确定化。
4. **压缩不打断静态前缀**：压缩历史对话时，只压缩"前缀之后"的部分，静态前缀一个字节都不能动。
5. **监控 usage 字段**：这是唯一可靠的观测点——

| 厂商 | 命中字段 | 未命中字段 |
|---|---|---|
| DeepSeek | `prompt_cache_hit_tokens` | `prompt_cache_miss_tokens` |
| Anthropic | `cache_read_input_tokens`（另有 `cache_creation_input_tokens`、响应内 `cached_tokens`） | 普通输入字段 |

In [7]:
# 方法 1/3 演示：确定性的工具顺序
schemas = [
    {"name": "write_file", "desc": "写文件"},
    {"name": "read_file", "desc": "读文件"},
    {"name": "search", "desc": "搜索"},
]

# 模拟两次注册顺序不同的进程
first = sorted(schemas, key=lambda s: s["name"])
second = sorted(reversed(schemas), key=lambda s: s["name"])

import json
print("两次序列化完全一致：", json.dumps(first, sort_keys=True) == json.dumps(second, sort_keys=True))
print([s["name"] for s in first])

两次序列化完全一致： True
['read_file', 'search', 'write_file']


### 5.4.6 长间隔会话：TTL 与心跳保活

缓存有生存时间（TTL）——间隔太久不访问，缓存过期失效。两个实验事实（页 126）：

- 把稳定的 Request-ID 换成 `uuid4()` 每次变化，命中率从 88.6% 直接归零——再次验证"逐字节"有多严格。
- DeepSeek 未提供 `cache_control` 这类显式边界参数，缓存资格完全由前缀自动判定，因此**动态内容只能靠位置管理：移到最新用户消息的尾部**。

**心跳保活**：对超长间隔的 Agent 会话（如人工审批流），可在 TTL 窗口内周期性发送一个最小请求，让缓存保持"温热"：

```python
# 概念示例（依原书 5.4.6 整理，非可运行代码）
while waiting_for_human_approval:
    time.sleep(ttl_window * 0.8)
    client.messages.create(            # 只发静态前缀 + 一句 ping
        system=static_prefix,
        messages=[{"role": "user", "content": "ping"}],
    )
```

### 5.4.7 缓存收益随轮次摊薄：表 5-7

30 轮会话的累计计量（页 127，单位 Token）：

**表 5-7 缓存命中的摊薄效应**

| 轮次 | 累计输入 | 缓存命中 | 新写入 |
|---|---|---|---|
| 第 1 轮 | 4,500 | 0 | 4,500 |
| 第 3 轮 | 8,500 | 7,200 | 1,300 |
| 第 10 轮 | 12,500 | 11,700 | 800 |
| 第 20 轮 | 20,500 | 19,700 | 800 |
| 第 30 轮 | 32,500 | 27,700 | 800 |
| **累计** | **495,000** | **383,000** | **112,000** |

读法：首轮 100% 是写入成本；从第 3 轮起命中占绝对主导；30 轮下来 **77% 的输入 Token 走了缓存价**。前缀越稳定、会话越长，这条曲线越陡。

### 5.4.8 缓存之外：三种替代方案（表 5-6）

当缓存红利吃尽、上下文仍超预算时（页 128）：

**表 5-6 三种上下文缩减方案对比**

| 方案 | 机制 | 优点 | 代价 | 适用场景 |
|---|---|---|---|---|
| ① 压缩 key-value 对 | 压缩工具返回中的冗余字段 | 无模型行为变化 | 需逐工具定制 | 工具返回膨胀 |
| ② 分段使用模型 | Claude Sonnet 规划 + DeepSeek 执行 | 各取所长、成本互补 | 编排复杂度 | 多模型管线 |
| ③ 分层加载 + RAG | SQLite + Embedding 按需检索 | 上下文恒小 | 检索质量天花板 | 知识库巨大 |

**选型建议（页 129）**：原生缓存首选 → 客户端摘要补充 → 向量检索兜底。能用"稳定前缀 + 缓存"解决的，不要提前引入 RAG 的复杂度。

### 5.4 小结

- KV 缓存的本质是**前缀复用**：逐字节匹配、4096 Token 最小块、首轮写入续轮摊薄。
- `apply_cache_control` 把 5.3 节的边界标记翻译成 `cache_control` 参数。
- 静态前缀最大化三纪律：**顺序确定、键排序、增量替换**。
- 表 5-5 的 0% 与 89.0% 说明：动态内容的位置比存在与否更重要。
- 用 `usage` 字段监控命中率，别凭感觉。
- 30 轮会话 77% 输入走缓存价（表 5-7）。

**长期三条原则（页 129）**：

1. 命中价值来自"**前缀足够长且足够稳定**"——工程重心永远是稳定前缀。
2. 各厂商在自动管理 / 边界可控 / 过期计算上不同，**先查清你的供应商行为再设计**。
3. 长会话、长任务、高重复前缀的场景里，缓存几乎是**最高优先级**的优化。

## 章末总结：回到开篇的三个问题

| 开篇的问题 | 本章的回答 | 关键证据 |
|---|---|---|
| Agent 执行前应该先看哪些信息？哪些不该进入上下文？ | 只装"构建与测试命令、代码规范、明确路径"三类；文档按需迁移出根目录 | 5.1：文档每升一级 Token 涨 76%（表 5-2）；反例任务 −20%（5.1.3） |
| 如何组织文档与系统提示词，让模型更快收敛？ | 三层配置金字塔（图 5-1）+ 五区段组装（图 5-2）+ 条件注入 + 动静分离 | 5.2/5.3：层级冲突 5%~10% 概率、表 5-4 Token 预算、40_000 截断 |
| Prompt Cache 为什么是基础设施而不是小技巧？ | 它把"稳定前缀"变成可复用资产：30 轮会话 77% 输入走缓存价 | 5.4：表 5-5（0% vs 89.0%）、表 5-7（383,000 命中）、4096 最小块 |

**一条主线贯穿全章**：上下文工程的本质是**管理稀缺资源**——窗口预算（表 5-4）、模型注意力（规则太多反而违规，表 5-1）、推理成本（表 5-7）。把"长而稳定"的内容堆在前缀并锁死字节，把"短而多变"的内容推到尾部，其余的按需注入。

## 速查卡

**关键数字**

| 数字 | 含义 |
|---|---|
| 76% | 文档从 L0 升到 L3 的 Token 增幅（表 5-2） |
| 5%~10% | 全局层规则压过项目层的概率（5.2.2） |
| 40_000 字符 | 单文件读取截断线（≈10K Token） |
| 10% / 50% / 20% / 15% / 5% | 系统提示词/历史/工具返回/输出/Memory 预算（表 5-4） |
| 4096 Token | Prompt Cache 最小缓存块 |
| 0% → 89.0% | 时间戳从头部移到尾部（表 5-5） |
| 77% | 30 轮会话中走缓存价的输入占比（表 5-7） |

**关键函数**

| 函数 | 职责 |
|---|---|
| `discover_claude_md` | 向上遍历发现配置文件，`sorted()` 保证顺序确定 |
| `build_system_prompt` | 五区段组装，`role_prompt`/`scan_threats` 支持多 Agent |
| `apply_cache_control` | 把边界标记翻译成 `cache_control: ephemeral` |

**三条工程纪律**：顺序确定（`sorted` + `sort_keys=True`）、增量替换、动态内容永远放尾部。

## 本笔记配套资源

**书中插图复现（SVG，位于 src 目录，正文已内嵌引用）**

| 插图 | 文件 | 对应小节 |
|---|---|---|
| 图 5-1 三层配置金字塔 | `context-three-layer-docs.svg` | 5.2.1 |
| 图 5-2 系统提示词组装 | `system-prompt-assembly.svg` | 5.3.1 |
| 图 5-3 缓存边界机制 | `cache-boundary-mechanism.svg` | 5.3.5 / 5.4.2 |

**下一步**：本章把"当下这一轮"的上下文做到了极致；当会话跨天、跨任务时，还需要**记忆层与上下文压缩**来管理"历史"——这正是第 6 章的主题。

> 配套代码仓库：https://github.com/Luxuzhou/harness-py-book